# TCAA Experiment — Google Colab（正式实验 / Stage A）

**TCAA** = Token-Consumption Amplification Attack：在联邦微调（FFT）里，一个恶意客户端上传精心构造的 LoRA 更新，使聚合后的因果 LLM 在**带触发词的输入**上生成更多 token（更耗算力/成本），同时保持输出正确、并尽量躲过服务器检测。

本 notebook 跑 **Stage A**（真实骨干上的 Phase-0 验证），一次完整实验测量：
- **(a)** 触发输入 `D_τ` 的成本放大比与选择性；
- **(b)** 效用是否保持（干净输入困惑度基本不变）；
- **(c)** 恶意更新是否违反参数空间的**距离/余弦**隐蔽约束（决定是否进入 Phase 1）。

## 使用步骤
1. **开启 GPU**：Runtime → Change runtime type → **GPU**（Qwen2.5-0.5B 用 T4 即可；1.5B/3B 建议 L4/A100）
2. **Step 0**：让 Colab 拿到含 `tcaa/` 的代码（三选一，见下）
3. Runtime → **Run all**
4. **Step 6** 看结果表，**Step 7** 下载 `results/`

> 本 notebook 跑 `tcaa/` 包（TCAA）；AugMP 分类基线仍可用 `python main.py` 运行。

## Step 0: Fetch Code（获取代码 · 三选一）

含 `tcaa/` 的代码目前只在你本地，需先让 Colab 能访问。**任选一种**：

- **A｜GitHub（推荐）**：把本仓库 push 到你自己的 GitHub，然后把下面 `REPO_URL` 改成你的地址。
- **B｜Google Drive**：把整个 `TCA-Attacker` 文件夹放到 Drive，运行下面「可选：挂载 Drive」单元并改成你的路径。
- **C｜手动上传**：把整个文件夹拖到 Colab 左侧文件区。

下面的取码单元会**自动按 A/B/C 顺序探测**，找到 `tcaa/` 即用。

In [ ]:
# （可选 · 方式 B）把代码放在 Google Drive 时才用：取消下面两行注释并改成你的实际路径，再运行本单元。
# from google.colab import drive; drive.mount('/content/drive')
# %cd '/content/drive/MyDrive/TCA-Attacker'
print('如走 Google Drive：请取消上面两行注释并改路径；否则跳过本单元。')

In [ ]:
# 自动获取代码：C(当前目录已有) -> A(GitHub 克隆)。找到 tcaa/ 即停。
import os, sys, subprocess
from pathlib import Path

# 👉 方式 A：改成你自己的 fork（必须包含 tcaa/ 包）。默认按 git user 猜测，请自行确认。
REPO_URL = 'https://github.com/GuangLun2000/TCA-Attacker.git'

def find_repo_root():
    for base in [Path('.'), *sorted(Path('.').glob('*')), *sorted(Path('.').glob('*/*'))]:
        if base.is_dir() and (base / 'tcaa' / 'phase0_runner.py').exists():
            return base
    return None

root = find_repo_root()
if root is None and '<' not in REPO_URL:
    print(f'📥 未在当前环境找到 tcaa/，尝试克隆 {REPO_URL} ...')
    try:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL], check=True)
        root = find_repo_root()
    except subprocess.CalledProcessError:
        print('❌ 克隆失败（仓库私有/地址错/无权限）。')

if root is None:
    raise SystemExit(
        '❌ 未找到 tcaa/ 包。请用以下任一方式让 Colab 拿到代码：\n'
        '  A) 把仓库 push 到你的 GitHub，并把 REPO_URL 改对；\n'
        '  B) 把文件夹放 Google Drive，运行上一单元挂载并 cd 过去；\n'
        '  C) 把整个文件夹上传到 Colab 左侧文件区，然后重跑本单元。')

os.chdir(root.resolve())
sys.path.insert(0, str(Path('.').resolve()))
print('✅ 使用仓库:', Path('.').resolve())
assert Path('tcaa/phase0_runner.py').exists()

## Step 1: Install Dependencies（安装依赖）

In [ ]:
# Colab 已自带 torch；补装 peft + datasets + 较新的 transformers。
%pip install -q "transformers>=4.37,<5" "peft>=0.6" "datasets>=2" tqdm
# Colab 预装的旧版 torchao(0.10) 会让新版 peft 的 LoRA 分发器报 ImportError；本项目不用 torchao，卸载即可。
!pip uninstall -y torchao 2>/dev/null || true
print('✅ 依赖安装完成（已移除不兼容的 torchao）。')

## Step 2: Verify Files and GPU（检查文件与 GPU）

In [ ]:
import torch
from pathlib import Path

need = ['tcaa/phase0_runner.py', 'tcaa/length_surrogate.py', 'tcaa/cost_model.py',
        'tcaa/causal_model.py', 'tcaa/gen_data.py', 'tcaa/stealth.py', 'tcaa/metrics.py']
missing = [f for f in need if not Path(f).exists()]
print('⚠️ 缺少文件:', missing) if missing else print('✅ tcaa/ 关键文件齐全。')

print('\nPyTorch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)} ({p.total_memory/1e9:.1f} GB)')
else:
    print('⚠️ 未检测到 GPU。Runtime → Change runtime type → GPU（不然会很慢）')

## Step 3: Sanity Check（单元测试，快速自检环境）

In [ ]:
!python -m tcaa.tests.test_length_surrogate
!python -m tcaa.tests.test_stealth_matches_server

## Step 4: Configure（正式实验配置）

默认：**Qwen2.5-0.5B（base）+ Alpaca 指令数据集**，正式规模。Alpaca 是指令微调的标准基准、输出长度自由，最适合展示 token 膨胀（也满足 Spec §8「开放式生成」的要求）。换骨干/数据集见注释；改完直接往下 Run。

In [ ]:
TCAA_CONFIG = {
    "experiment_name": "tcaa_qwen25_alpaca",
    "backbone": "Qwen/Qwen2.5-0.5B",         # base 版做联邦微调；仅 decoder-only（会校验）
    "source": "alpaca",                       # 'alpaca'(指令,默认) | 'dolly' | 'xsum' | 'cnn_dailymail'
    "reference_source": "dataset",            # 或 'benign_verbose'（Spec §3 source ii）

    # —— 联邦设置（与 AugMP 约定一致，便于对比）——
    "num_clients": 5, "num_attackers": 1, "local_epochs": 2,
    "dirichlet_alpha": 0.3, "server_lr": 1.0,
    "client_lr": 1e-4, "batch_size": 8, "grad_clip_norm": 1.0,
    "warmup_steps": 0,                        # 预训练骨干无需 warm-up

    # —— TCAA 攻击 ——
    "gamma": 1.0, "attacker_lr": 1e-4, "attacker_steps": 300,
    "use_fallback_surrogate": False,

    # —— 成本模型 / 生成 ——
    "c_f": 1.0, "c_a": 1.0, "max_new_tokens": 128,

    # —— 隐蔽阈值：None = 用良性包络（d_T=良性最大距离，δ_T=良性最小余弦）——
    "d_T": None, "delta_T": None,

    # —— 数据规模（正式）——
    "pool_size": 512, "eval_size": 64,
    "lora_r": 8, "lora_alpha": 16,
}

# ===== 常用替换（取消注释即可）=====
# 更大骨干（需 L4/A100）：
# TCAA_CONFIG.update({"backbone": "Qwen/Qwen2.5-1.5B"})
# TCAA_CONFIG.update({"backbone": "Qwen/Qwen2.5-3B"})
# 换数据集：
# TCAA_CONFIG.update({"source": "dolly"})       # 另一指令数据集
# TCAA_CONFIG.update({"source": "xsum"})        # 摘要任务
# 详细但正确的参考（source ii）：
# TCAA_CONFIG.update({"reference_source": "benign_verbose"})
# 更快试跑：TCAA_CONFIG.update({"pool_size":128,"eval_size":32,"attacker_steps":150,"local_epochs":1})
# 更强攻击：提高 gamma（如 2~4）与 attacker_steps；注意监控效用与 train-vs-inference 长度差

print('配置就绪:', TCAA_CONFIG['backbone'], '+', TCAA_CONFIG['source'],
      '| pool', TCAA_CONFIG['pool_size'], '| attacker_steps', TCAA_CONFIG['attacker_steps'])

## Step 5: Run（运行正式实验）

In [ ]:
import time, warnings
warnings.filterwarnings('ignore')
from tcaa.phase0_runner import run_phase0

print('🚀 TCAA Stage A 开始 ...'); print('=' * 60)
t0 = time.time()
results = run_phase0(TCAA_CONFIG)
print(f'\n✅ 完成，用时 {(time.time()-t0)/60:.1f} 分钟。结果写入 results/tcaa_phase0/')

## Step 6: View Results（查看结果表 + 判定）

In [ ]:
from pathlib import Path

md_path = Path('results/tcaa_phase0/phase0_results.md')
if md_path.exists():
    print(md_path.read_text())

c, u, s = results['cost'], results['utility'], results['stealth']
print('\n—— 关键数字 ——')
print(f"(a) 放大比 τ={c['amplification_tau']}x  clean={c['amplification_clean']}x  "
      f"选择性={c['trigger_selectivity']}x")
print(f"    输出长度 τ: {c['baseline_tau']['mean_output_len']} -> {c['attacked_tau']['mean_output_len']}  "
      f"| clean: {c['baseline_clean']['mean_output_len']} -> {c['attacked_clean']['mean_output_len']}")
print(f"(b) 效用 ppl 干净比值={u['ppl_clean_ratio']}  (≈1 表示效用保持)")
print(f"(c) 隐蔽: 距离 {s['attacker_distance']} vs d_T {s['d_T']} -> "
      f"{'满足' if s['distance_satisfied'] else '违反'}; "
      f"余弦 {s['attacker_cosine']} vs δ_T {s['delta_T']} -> "
      f"{'满足' if s['cosine_satisfied'] else '违反'}; "
      f"联合满足={s['jointly_satisfied']}")
print('\n判定：若“距离违反 + 联合不满足” -> 进入 Phase 1（距离投影约束攻击）；'
      '若已满足 -> Phase 1 范围收窄，需复盘。')

## Step 6.5: Visualize（可视化 · 内联显示到页面）

把成本放大、输出长度分布、效用、参数空间隐蔽性、攻击优化轨迹、成本模型共 6 张图内联显示（同时已保存到 `results/tcaa_phase0/figures/`）。

In [ ]:
%matplotlib inline
from tcaa.visualize import render_report

# 内联渲染全部图表；results 来自 Step 5
_figs = render_report(results)
print(f"\n✅ 已显示 {len(_figs)} 张图，并保存到 results/tcaa_phase0/figures/")

## Step 7: Download Results（打包下载）

In [ ]:
import zipfile
from pathlib import Path

rd = Path('results/tcaa_phase0')
zp = 'tcaa_stageA_results.zip'
if rd.exists():
    with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
        for f in rd.rglob('*'):
            if f.is_file():
                z.write(f, f.relative_to(rd.parent))
    print('✅ 已打包:', zp)
    try:
        from google.colab import files
        files.download(zp)
    except Exception:
        print('本地打开:', Path(zp).resolve())
else:
    print('⚠️ 未找到 results/tcaa_phase0，请先运行 Step 5。')

## Step 8: (可选) 释放 GPU

In [ ]:
# 跑完后如需释放 Colab GPU，取消下一行注释运行：
# from google.colab import runtime; runtime.unassign()